# 02 - Data preparation
## 1. Setup and load

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

trips = pd.read_csv('../data/trips_all_cities_master.csv')
trips['date'] = pd.to_datetime(trips['date'])

## 2. Hourly aggregation
The target is the number of trips per city and hour, not single trips

In [2]:
demand = (trips.groupby(['city', 'date', 'hour']).size()
               .rename('trips').reset_index())

## Complete grid and zero fills
hours without trips are missing from raw data. Lags below only work if every hour exists.

In [3]:
parts = []
for city, d in trips.groupby('city'):
    days = pd.date_range(d['date'].min(), d['date'].max(), freq='D')
    parts.append(pd.MultiIndex.from_product(
        [[city], days, range(24)], names=['city', 'date', 'hour']
    ).to_frame(index=False))
grid = pd.concat(parts, ignore_index=True)

demand = grid.merge(demand, on=['city', 'date', 'hour'], how='left')
demand['trips'] = demand['trips'].fillna(0).astype(int)

## 3. Sanity checks

In [4]:
print('Rows:', len(demand), '(expected 16128)')
print('Total trips:', demand['trips'].sum(), '(raw:', len(trips), ')')
print('Zero-trip hours:', (demand['trips'] == 0).sum())

Rows: 16128 (expected 16128)
Total trips: 393182 (raw: 393182 )
Zero-trip hours: 1049
